# NB11 — Full Training and One-Time Official Test Evaluation

## Dissertation
**Explainable and Trustworthy Multimodal Deep Learning for Predictive Maintenance of Industrial Assets**

## Purpose

This notebook performs the final held-out evaluation on NASA C-MAPSS FD001.
The experimental configuration, model architectures, preprocessing choices and neural training epochs are frozen before the official test labels are accessed.

The protocol is intentionally divided into two stages:

1. **Pre-test stage:** derive the final neural epoch counts from NB10, prepare the full 100-engine training set, train and save the four frozen models, and create a pre-test freeze manifest.
2. **Official-test stage:** after explicit authorisation, load the 100 FD001 test engines and official endpoint RUL labels, generate exactly one prediction per engine for each model, and save the final metrics and predictions.

The official test results are not used for tuning, model selection, epoch adjustment or preprocessing changes. Any post-test analysis is descriptive only.

## Section 0 — Colab and Project Setup

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Dissertation/Project/dissertation-rul-xai'
else:
    BASE = os.path.abspath('..')

assert os.path.isdir(BASE), f'Project root not found: {BASE}'
assert os.path.isfile(f'{BASE}/data/raw/CMAPSS/train_FD001.txt'), 'train_FD001.txt not found'
assert os.path.isfile(f'{BASE}/data/raw/CMAPSS/test_FD001.txt'), 'test_FD001.txt not found'
assert os.path.isfile(f'{BASE}/data/raw/CMAPSS/RUL_FD001.txt'), 'RUL_FD001.txt not found'

RAW_DIR          = f'{BASE}/data/raw/CMAPSS'
FV_REPORT_DIR    = f'{BASE}/reports/final_validation'
FULL_DATA_DIR    = f'{BASE}/data/processed/final_full_training'
FINAL_MODEL_DIR  = f'{BASE}/models/final'
FINAL_TEST_DIR   = f'{BASE}/reports/final_test'

for path in [FULL_DATA_DIR, FINAL_MODEL_DIR, FINAL_TEST_DIR]:
    os.makedirs(path, exist_ok=True)

print(f'Project root: {BASE}')
print(f'Full-training data: {FULL_DATA_DIR}')
print(f'Final models: {FINAL_MODEL_DIR}')
print(f'Final-test reports: {FINAL_TEST_DIR}')

## Section 1 — Environment Manifest and Imports

In [ ]:
import gc
import json
import random
import hashlib
import platform
import subprocess
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
import xgboost as xgb

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    root_mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
    r2_score,
)
from xgboost import XGBRegressor
from tensorflow.keras import layers, models

# Attempt deterministic execution.
deterministic_enabled = False
try:
    tf.config.experimental.enable_op_determinism()
    deterministic_enabled = True
except Exception as exc:
    print(f'enable_op_determinism not available: {exc}')

gpus = tf.config.list_physical_devices('GPU')
gpu_model = 'None'
if gpus:
    try:
        gpu_model = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            text=True,
        ).strip()
    except Exception:
        gpu_model = str(gpus[0])

env_manifest = {
    'recorded_at': datetime.now(timezone.utc).isoformat(),
    'python': platform.python_version(),
    'tensorflow': tf.__version__,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit_learn': sklearn.__version__,
    'xgboost': xgb.__version__,
    'gpu_available': bool(gpus),
    'gpu_model': gpu_model,
    'deterministic_ops': deterministic_enabled,
    'platform': platform.platform(),
}

with open(f'{FINAL_TEST_DIR}/experiment_environment.json', 'w') as f:
    json.dump(env_manifest, f, indent=2)

print('Environment manifest:')
for key, value in env_manifest.items():
    print(f'  {key}: {value}')

## Section 2 — Frozen Official-Test Protocol

The following values are carried forward from NB10 and must not be changed after viewing the official test results.

- Target: capped RUL regression, cap = 125 cycles
- Sequence window: 30 cycles
- Derived rolling window: 5 cycles
- Raw sensor view: 14 variable sensors
- Derived view: 42 rolling/delta features plus `cycle_index`
- Model seed: 42 for every neural model
- Final neural epochs: calculated from the median NB10 best epoch across split seeds 21, 42 and 84
- XGBoost: frozen NB04/NB10 configuration
- Test evaluation: one endpoint per test engine, 100 engines in total

In [ ]:
MODEL_SEED = 42
RUL_CAP     = 125
WINDOW_SIZE = 30
ROLLING_WIN = 5
STRIDE      = 1
SPLIT_SEEDS = [21, 42, 84]

SENSOR_COLS = [
    'sensor_measurement_11', 'sensor_measurement_4',  'sensor_measurement_12',
    'sensor_measurement_7',  'sensor_measurement_15', 'sensor_measurement_21',
    'sensor_measurement_20', 'sensor_measurement_2',  'sensor_measurement_17',
    'sensor_measurement_3',  'sensor_measurement_8',  'sensor_measurement_13',
    'sensor_measurement_9',  'sensor_measurement_14',
]
TARGET_COL = 'RUL_capped'

XGB_PARAMS = dict(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=MODEL_SEED,
    n_jobs=-1,
)

GRU_BATCH = 256
MLP_BATCH = 128
GRU_LR    = 0.001
MLP_LR    = 0.001

# Execution controls.
OVERWRITE_FINAL_MODELS = False
AUTHORISE_OFFICIAL_TEST = False   # Change to True only after reviewing the pre-test freeze manifest.
OVERWRITE_OFFICIAL_TEST = False   # Keep False. Existing official-test outputs are loaded, not replaced.

assert len(SENSOR_COLS) == 14

frozen_protocol = {
    'MODEL_SEED': MODEL_SEED,
    'RUL_CAP': RUL_CAP,
    'WINDOW_SIZE': WINDOW_SIZE,
    'ROLLING_WIN': ROLLING_WIN,
    'STRIDE': STRIDE,
    'SPLIT_SEEDS_FOR_EPOCH_SELECTION': SPLIT_SEEDS,
    'SENSOR_COLS': SENSOR_COLS,
    'prediction_clipping': [0, RUL_CAP],
    'XGB_PARAMS': XGB_PARAMS,
    'GRU_BATCH': GRU_BATCH,
    'MLP_BATCH': MLP_BATCH,
    'GRU_LR': GRU_LR,
    'MLP_LR': MLP_LR,
    'official_test_unit_of_evaluation': 'one endpoint prediction per engine',
    'official_test_expected_engines': 100,
}

with open(f'{FINAL_TEST_DIR}/frozen_official_test_protocol.json', 'w') as f:
    json.dump(frozen_protocol, f, indent=2)

print('Frozen protocol saved.')
print(f'AUTHORISE_OFFICIAL_TEST = {AUTHORISE_OFFICIAL_TEST}')

## Section 3 — Derive and Freeze Final Epoch Counts from NB10

The epoch counts are derived only from repeated-validation artefacts. The official test data and labels are not accessed in this section.

Rule:

- Use the median best epoch across split seeds 21, 42 and 84.
- If at least two runs hit the configured maximum epoch, retain that maximum.
- Freeze the selected values before full-data training and official-test evaluation.

In [ ]:
MAX_EPOCHS = {
    'GRU': 30,
    'DerivedOnlyMLP': 60,
    'MultiViewGRUFusion': 60,
}
NEURAL_MODELS = list(MAX_EPOCHS)

epoch_rows = []
for seed in SPLIT_SEEDS:
    metrics_path = f'{FV_REPORT_DIR}/split_seed_{seed}/model_metrics.csv'
    assert os.path.isfile(metrics_path), f'Missing NB10 metrics: {metrics_path}'
    split_metrics = pd.read_csv(metrics_path)

    for model_name in NEURAL_MODELS:
        row = split_metrics.loc[split_metrics['model'] == model_name]
        assert len(row) == 1, f'Expected one {model_name} row for seed {seed}'
        best_epoch = row.iloc[0]['best_epoch']
        assert pd.notna(best_epoch), f'Missing best_epoch: seed={seed}, model={model_name}'
        epoch_rows.append({
            'split_seed': seed,
            'model': model_name,
            'best_epoch': int(best_epoch),
        })

epoch_evidence = pd.DataFrame(epoch_rows)
selection_rows = []
FINAL_EPOCHS = {}

for model_name in NEURAL_MODELS:
    values = epoch_evidence.loc[
        epoch_evidence['model'] == model_name, 'best_epoch'
    ].astype(int).tolist()
    max_epoch = MAX_EPOCHS[model_name]
    hit_max_count = sum(v == max_epoch for v in values)

    if hit_max_count >= 2:
        selected = max_epoch
        rule = 'maximum retained because at least two splits hit maximum'
    else:
        selected = int(np.median(values))
        rule = 'median best epoch across split seeds 21, 42 and 84'

    FINAL_EPOCHS[model_name] = selected
    selection_rows.append({
        'model': model_name,
        'seed_21_best_epoch': values[0],
        'seed_42_best_epoch': values[1],
        'seed_84_best_epoch': values[2],
        'configured_max_epoch': max_epoch,
        'selection_rule': rule,
        'final_epoch': selected,
    })

epoch_selection = pd.DataFrame(selection_rows)

# Frozen values confirmed in NB10.
assert FINAL_EPOCHS == {
    'GRU': 12,
    'DerivedOnlyMLP': 59,
    'MultiViewGRUFusion': 12,
}, f'Unexpected final epoch selection: {FINAL_EPOCHS}'

epoch_path = f'{FINAL_TEST_DIR}/final_epoch_selection_fd001.csv'
epoch_selection.to_csv(epoch_path, index=False)

print(epoch_selection.to_string(index=False))
print(f'\nFinal epoch selection saved: {epoch_path}')

## Section 4 — Load Full FD001 Training Data

Only `train_FD001.txt` is read here. Test trajectories and official RUL labels remain unopened until the models are trained and frozen.

In [ ]:
INDEX_COLS = ['unit_number', 'time_in_cycles']
OP_COLS = [f'operational_setting_{i}' for i in range(1, 4)]
ALL_SENSOR_COLS = [f'sensor_measurement_{i}' for i in range(1, 22)]
ALL_COLS = INDEX_COLS + OP_COLS + ALL_SENSOR_COLS

raw_train = pd.read_csv(
    f'{RAW_DIR}/train_FD001.txt',
    sep=r'\s+',
    header=None,
    names=ALL_COLS,
)

max_cycles = raw_train.groupby('unit_number')['time_in_cycles'].max().rename('max_cycle')
raw_train = raw_train.join(max_cycles, on='unit_number')
raw_train['RUL'] = raw_train['max_cycle'] - raw_train['time_in_cycles']
raw_train['RUL_capped'] = raw_train['RUL'].clip(upper=RUL_CAP).astype(float)
raw_train.drop(columns=['max_cycle'], inplace=True)

train_units = sorted(raw_train['unit_number'].unique().tolist())

assert len(train_units) == 100, f'Expected 100 training engines, got {len(train_units)}'
assert train_units == list(range(1, 101)), 'Unexpected training engine identifiers'
assert not raw_train.duplicated(['unit_number', 'time_in_cycles']).any()
assert raw_train[SENSOR_COLS + ['RUL', 'RUL_capped']].notna().all().all()

print(f'Full training rows: {len(raw_train):,}')
print(f'Training engines: {len(train_units)}')
print(f'Cycle range: {raw_train.time_in_cycles.min()} to {raw_train.time_in_cycles.max()}')
print(f'Capped RUL range: {raw_train.RUL_capped.min():.0f} to {raw_train.RUL_capped.max():.0f}')

## Section 5 — Shared Preprocessing and Model Builders

These functions reproduce the NB10 feature engineering, scaling, window construction and frozen architectures.

In [ ]:
def compute_derived_features(df, sensor_features, rolling_window=ROLLING_WIN):
    """Per-engine rolling mean, rolling standard deviation, delta and cycle_index."""
    parts = []
    for _, unit_df in df.groupby('unit_number'):
        unit_df = unit_df.sort_values('time_in_cycles').copy()
        for sensor in sensor_features:
            unit_df[f'{sensor}_rmean'] = (
                unit_df[sensor].rolling(rolling_window, min_periods=1).mean()
            )
            unit_df[f'{sensor}_rstd'] = (
                unit_df[sensor].rolling(rolling_window, min_periods=1).std().fillna(0)
            )
            unit_df[f'{sensor}_delta'] = unit_df[sensor] - unit_df[sensor].iloc[0]
        unit_df['cycle_index'] = unit_df['time_in_cycles']
        parts.append(unit_df)
    return pd.concat(parts, ignore_index=True)


def create_multiview_windows(b_df, c_df, sensor_cols, derived_cols, target_col,
                             window_size=WINDOW_SIZE, stride=STRIDE):
    """Create paired full-training sequence and derived-feature windows."""
    c_lookup = c_df.set_index(['unit_number', 'time_in_cycles'])
    X_seq, X_der, y, meta = [], [], [], []

    for unit, unit_df in b_df.groupby('unit_number'):
        unit_df = unit_df.sort_values('time_in_cycles').reset_index(drop=True)
        features = unit_df[sensor_cols].values.astype(np.float32)
        targets = unit_df[target_col].values.astype(np.float32)
        cycles = unit_df['time_in_cycles'].values
        raw_rul = unit_df['RUL'].values

        if len(unit_df) < window_size:
            continue

        for end in range(window_size - 1, len(unit_df), stride):
            start = end - window_size + 1
            cycle = cycles[end]
            key = (unit, cycle)
            assert key in c_lookup.index

            X_seq.append(features[start:end + 1])
            X_der.append(c_lookup.loc[key, derived_cols].values.astype(np.float32))
            y.append(targets[end])
            meta.append({
                'unit_number': unit,
                'time_in_cycles': cycle,
                'RUL': raw_rul[end],
                'RUL_capped': targets[end],
            })

    return (
        np.asarray(X_seq, dtype=np.float32),
        np.asarray(X_der, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        pd.DataFrame(meta),
    )


def build_gru(window_size, n_sensors):
    model = models.Sequential([
        layers.Input(shape=(window_size, n_sensors)),
        layers.GRU(64, return_sequences=True),
        layers.GRU(32),
        layers.Dense(50, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1),
    ], name='GRU')
    model.compile(optimizer=tf.keras.optimizers.Adam(GRU_LR), loss='mse', metrics=['mae'])
    return model


def build_derived_mlp(n_features):
    inp = layers.Input(shape=(n_features,), name='degradation_feature_view')
    x = layers.Dense(64, activation='relu')(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1, name='rul_prediction')(x)
    model = models.Model(inp, out, name='DerivedOnlyMLP')
    model.compile(optimizer=tf.keras.optimizers.Adam(MLP_LR), loss='mse', metrics=['mae'])
    return model


def build_multiview_gru(window_size, n_sensors, n_derived):
    seq_in = layers.Input(shape=(window_size, n_sensors), name='sensor_sequence_view')
    seq_x = layers.GRU(64, return_sequences=False, name='sensor_gru_encoder')(seq_in)
    seq_x = layers.Dropout(0.2)(seq_x)

    der_in = layers.Input(shape=(n_derived,), name='degradation_feature_view')
    der_x = layers.Dense(64, activation='relu', name='degradation_dense_1')(der_in)
    der_x = layers.Dropout(0.2)(der_x)
    der_x = layers.Dense(32, activation='relu', name='degradation_dense_2')(der_x)

    fused = layers.Concatenate(name='view_fusion')([seq_x, der_x])
    z = layers.Dense(64, activation='relu', name='fusion_dense_1')(fused)
    z = layers.Dropout(0.2)(z)
    z = layers.Dense(32, activation='relu', name='fusion_dense_2')(z)
    out = layers.Dense(1, name='rul_prediction')(z)

    model = models.Model([seq_in, der_in], out, name='MultiViewGRUFusion')
    model.compile(optimizer=tf.keras.optimizers.Adam(MLP_LR), loss='mse', metrics=['mae'])
    return model


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


print('Shared preprocessing and model builders defined.')

## Section 6 — Prepare Full-Training Features and Windows

Both scalers are fitted on the complete FD001 training set only. The same fitted scalers will later be applied to test trajectories without refitting.

In [ ]:
train_derived = compute_derived_features(raw_train, SENSOR_COLS)

DERIVED_COLS = []
for sensor in SENSOR_COLS:
    DERIVED_COLS.extend([
        f'{sensor}_rmean',
        f'{sensor}_rstd',
        f'{sensor}_delta',
    ])
DERIVED_COLS.append('cycle_index')
FEATURE_SET_C = SENSOR_COLS + DERIVED_COLS

assert len(DERIVED_COLS) == 43
assert len(FEATURE_SET_C) == 57
assert set(DERIVED_COLS).issubset(train_derived.columns)
assert train_derived[FEATURE_SET_C].notna().all().all()

scaler_b_path = f'{FULL_DATA_DIR}/scaler_feature_set_b_fd001.joblib'
scaler_c_path = f'{FULL_DATA_DIR}/scaler_feature_set_c_fd001.joblib'
feature_manifest_path = f'{FULL_DATA_DIR}/feature_manifest_fd001.json'

if (os.path.isfile(scaler_b_path) and os.path.isfile(scaler_c_path)
        and not OVERWRITE_FINAL_MODELS):
    scaler_b = joblib.load(scaler_b_path)
    scaler_c = joblib.load(scaler_c_path)
    print('Existing full-training scalers retained.')
else:
    scaler_b = StandardScaler().fit(train_derived[SENSOR_COLS])
    scaler_c = StandardScaler().fit(train_derived[FEATURE_SET_C])
    joblib.dump(scaler_b, scaler_b_path)
    joblib.dump(scaler_c, scaler_c_path)
    print('Full-training scalers fitted and saved.')

train_b = train_derived.copy()
train_c = train_derived.copy()
train_b[SENSOR_COLS] = scaler_b.transform(train_derived[SENSOR_COLS])
train_c[FEATURE_SET_C] = scaler_c.transform(train_derived[FEATURE_SET_C])

X_seq_train, X_der_train, y_train, meta_train = create_multiview_windows(
    train_b,
    train_c,
    SENSOR_COLS,
    DERIVED_COLS,
    TARGET_COL,
)

assert X_seq_train.shape[1:] == (WINDOW_SIZE, len(SENSOR_COLS))
assert X_der_train.shape[1] == len(DERIVED_COLS)
assert len(X_seq_train) == len(X_der_train) == len(y_train) == len(meta_train)
assert meta_train['unit_number'].nunique() == 100
assert np.isfinite(X_seq_train).all()
assert np.isfinite(X_der_train).all()
assert np.isfinite(y_train).all()

feature_manifest = {
    'sensor_cols': SENSOR_COLS,
    'derived_cols': DERIVED_COLS,
    'feature_set_c': FEATURE_SET_C,
    'n_sensor_features': len(SENSOR_COLS),
    'n_derived_features': len(DERIVED_COLS),
    'n_feature_set_c': len(FEATURE_SET_C),
    'rolling_window': ROLLING_WIN,
    'sequence_window': WINDOW_SIZE,
}
if not os.path.isfile(feature_manifest_path) or OVERWRITE_FINAL_MODELS:
    with open(feature_manifest_path, 'w') as f:
        json.dump(feature_manifest, f, indent=2)
else:
    with open(feature_manifest_path) as f:
        saved_feature_manifest = json.load(f)
    assert saved_feature_manifest == feature_manifest, (
        'Saved feature manifest differs from frozen protocol'
    )

print(f'Full-training rows for XGBoost: {len(train_c):,}')
print(f'Full-training neural windows: {len(y_train):,}')
print(f'X_seq_train: {X_seq_train.shape}')
print(f'X_der_train: {X_der_train.shape}')
print('Full-training scalers and feature manifest available.')

## Section 7 — Train and Save Frozen Models on All 100 Training Engines

The neural models are trained for the fixed epoch counts selected in Section 3. No validation data, early stopping or post-test adjustment is used during this refit.

Each neural model receives an independent seed reset before construction. Model artefacts and training histories are saved immediately so an interrupted Colab session can resume model by model.

In [ ]:
MODEL_PATHS = {
    'XGBoost': f'{FINAL_MODEL_DIR}/XGBoost_feature_set_c_fd001.joblib',
    'GRU': f'{FINAL_MODEL_DIR}/GRU_B_window30_fd001.keras',
    'DerivedOnlyMLP': f'{FINAL_MODEL_DIR}/DerivedOnlyMLP_window30_fd001.keras',
    'MultiViewGRUFusion': f'{FINAL_MODEL_DIR}/MultiViewGRUFusion_window30_fd001.keras',
}
HISTORY_PATHS = {
    'GRU': f'{FINAL_TEST_DIR}/full_training_history_GRU_fd001.csv',
    'DerivedOnlyMLP': f'{FINAL_TEST_DIR}/full_training_history_DerivedOnlyMLP_fd001.csv',
    'MultiViewGRUFusion': f'{FINAL_TEST_DIR}/full_training_history_MultiViewGRUFusion_fd001.csv',
}

# XGBoost.
if os.path.isfile(MODEL_PATHS['XGBoost']) and not OVERWRITE_FINAL_MODELS:
    print('[XGBoost] Existing final model retained.')
else:
    print('[XGBoost] Training on all full-training rows...')
    xgb_model = XGBRegressor(**XGB_PARAMS)
    xgb_model.fit(train_c[FEATURE_SET_C], train_c[TARGET_COL])
    joblib.dump(xgb_model, MODEL_PATHS['XGBoost'])
    print(f'[XGBoost] Saved: {MODEL_PATHS["XGBoost"]}')

# GRU.
if (os.path.isfile(MODEL_PATHS['GRU']) and os.path.isfile(HISTORY_PATHS['GRU'])
        and not OVERWRITE_FINAL_MODELS):
    print('[GRU] Existing final model and history retained.')
else:
    print(f'[GRU] Training for frozen {FINAL_EPOCHS["GRU"]} epochs...')
    tf.keras.backend.clear_session(); gc.collect()
    tf.keras.utils.set_random_seed(MODEL_SEED)
    gru_model = build_gru(WINDOW_SIZE, len(SENSOR_COLS))
    gru_history = gru_model.fit(
        X_seq_train,
        y_train,
        epochs=FINAL_EPOCHS['GRU'],
        batch_size=GRU_BATCH,
        shuffle=True,
        verbose=1,
    )
    gru_model.save(MODEL_PATHS['GRU'])
    pd.DataFrame(gru_history.history).to_csv(HISTORY_PATHS['GRU'], index=False)
    del gru_model, gru_history
    tf.keras.backend.clear_session(); gc.collect()

# Derived-only MLP.
if (os.path.isfile(MODEL_PATHS['DerivedOnlyMLP'])
        and os.path.isfile(HISTORY_PATHS['DerivedOnlyMLP'])
        and not OVERWRITE_FINAL_MODELS):
    print('[DerivedOnlyMLP] Existing final model and history retained.')
else:
    print(f'[DerivedOnlyMLP] Training for frozen {FINAL_EPOCHS["DerivedOnlyMLP"]} epochs...')
    tf.keras.backend.clear_session(); gc.collect()
    tf.keras.utils.set_random_seed(MODEL_SEED)
    mlp_model = build_derived_mlp(len(DERIVED_COLS))
    mlp_history = mlp_model.fit(
        X_der_train,
        y_train,
        epochs=FINAL_EPOCHS['DerivedOnlyMLP'],
        batch_size=MLP_BATCH,
        shuffle=True,
        verbose=1,
    )
    mlp_model.save(MODEL_PATHS['DerivedOnlyMLP'])
    pd.DataFrame(mlp_history.history).to_csv(HISTORY_PATHS['DerivedOnlyMLP'], index=False)
    del mlp_model, mlp_history
    tf.keras.backend.clear_session(); gc.collect()

# Multi-view fusion.
if (os.path.isfile(MODEL_PATHS['MultiViewGRUFusion'])
        and os.path.isfile(HISTORY_PATHS['MultiViewGRUFusion'])
        and not OVERWRITE_FINAL_MODELS):
    print('[MultiViewGRUFusion] Existing final model and history retained.')
else:
    print(f'[MultiViewGRUFusion] Training for frozen {FINAL_EPOCHS["MultiViewGRUFusion"]} epochs...')
    tf.keras.backend.clear_session(); gc.collect()
    tf.keras.utils.set_random_seed(MODEL_SEED)
    fusion_model = build_multiview_gru(
        WINDOW_SIZE,
        len(SENSOR_COLS),
        len(DERIVED_COLS),
    )
    fusion_history = fusion_model.fit(
        [X_seq_train, X_der_train],
        y_train,
        epochs=FINAL_EPOCHS['MultiViewGRUFusion'],
        batch_size=MLP_BATCH,
        shuffle=True,
        verbose=1,
    )
    fusion_model.save(MODEL_PATHS['MultiViewGRUFusion'])
    pd.DataFrame(fusion_history.history).to_csv(HISTORY_PATHS['MultiViewGRUFusion'], index=False)
    del fusion_model, fusion_history
    tf.keras.backend.clear_session(); gc.collect()

for model_name, model_path in MODEL_PATHS.items():
    assert os.path.isfile(model_path), f'Missing final model: {model_name}'
for model_name, history_path in HISTORY_PATHS.items():
    assert os.path.isfile(history_path), f'Missing full-training history: {model_name}'

print('\nAll four frozen models are available.')

## Section 8 — Pre-Test Freeze Manifest

This manifest proves that model files, scalers, feature definitions and final epoch counts existed before the official test labels were read.

Review this section before changing `AUTHORISE_OFFICIAL_TEST` to `True`.

In [ ]:
FREEZE_FILES = {
    **{f'model_{name}': path for name, path in MODEL_PATHS.items()},
    **{f'history_{name}': path for name, path in HISTORY_PATHS.items()},
    'scaler_feature_set_b': f'{FULL_DATA_DIR}/scaler_feature_set_b_fd001.joblib',
    'scaler_feature_set_c': f'{FULL_DATA_DIR}/scaler_feature_set_c_fd001.joblib',
    'feature_manifest': f'{FULL_DATA_DIR}/feature_manifest_fd001.json',
    'epoch_selection': f'{FINAL_TEST_DIR}/final_epoch_selection_fd001.csv',
    'frozen_protocol': f'{FINAL_TEST_DIR}/frozen_official_test_protocol.json',
}

for label, path in FREEZE_FILES.items():
    assert os.path.isfile(path), f'Pre-test freeze file missing: {label} -> {path}'

current_file_records = {
    label: {
        'path': os.path.relpath(path, BASE),
        'sha256': sha256_file(path),
        'size_bytes': os.path.getsize(path),
    }
    for label, path in FREEZE_FILES.items()
}

pretest_manifest_path = f'{FINAL_TEST_DIR}/pretest_freeze_manifest.json'
if os.path.isfile(pretest_manifest_path) and not OVERWRITE_FINAL_MODELS:
    with open(pretest_manifest_path) as f:
        pretest_manifest = json.load(f)
    assert pretest_manifest['files'] == current_file_records, (
        'Existing pre-test manifest does not match current frozen artefacts'
    )
    assert pretest_manifest['final_epochs'] == FINAL_EPOCHS
    print('Existing pre-test freeze manifest retained and verified.')
else:
    pretest_manifest = {
        'created_at': datetime.now(timezone.utc).isoformat(),
        'status': 'models and preprocessing frozen before official test-label access',
        'train_engines': 100,
        'training_rows_xgboost': int(len(train_c)),
        'training_windows_neural': int(len(y_train)),
        'final_epochs': FINAL_EPOCHS,
        'files': current_file_records,
    }
    with open(pretest_manifest_path, 'w') as f:
        json.dump(pretest_manifest, f, indent=2)
    print('New pre-test freeze manifest created.')

print('=== PRE-TEST FREEZE COMPLETE ===')
print(f'Frozen epochs: {FINAL_EPOCHS}')
print(f'Model artefacts: {len(MODEL_PATHS)}')
print(f'Manifest: {pretest_manifest_path}')
print('\nOfficial test labels have not been read by this notebook up to this point.')
print('Review the manifest, then set AUTHORISE_OFFICIAL_TEST=True in Section 2.')

## Section 9 — Official Test Endpoint Construction and Evaluation Functions

The FD001 test trajectories stop before failure. `RUL_FD001.txt` provides the remaining life at the final observed cycle of each test engine. Therefore, this notebook creates exactly one endpoint sample per engine:

- the last 30-cycle scaled sensor sequence;
- the derived-feature vector at the final observed cycle;
- the frozen Feature Set C vector for XGBoost;
- the capped official endpoint RUL target.

In [ ]:
def build_test_endpoints(raw_test, official_rul, scaler_b, scaler_c):
    test_units = sorted(raw_test['unit_number'].unique().tolist())

    assert len(test_units) == 100, f'Expected 100 test engines, got {len(test_units)}'
    assert test_units == list(range(1, 101)), 'Unexpected test engine identifiers'
    assert len(official_rul) == 100, f'Expected 100 official RUL labels, got {len(official_rul)}'
    assert not raw_test.duplicated(['unit_number', 'time_in_cycles']).any()

    target_map = {
        unit: float(official_rul.iloc[index])
        for index, unit in enumerate(test_units)
    }

    test_derived = compute_derived_features(raw_test, SENSOR_COLS)
    assert test_derived[FEATURE_SET_C].notna().all().all()

    test_b = test_derived.copy()
    test_c = test_derived.copy()
    test_b[SENSOR_COLS] = scaler_b.transform(test_derived[SENSOR_COLS])
    test_c[FEATURE_SET_C] = scaler_c.transform(test_derived[FEATURE_SET_C])

    c_lookup = test_c.set_index(['unit_number', 'time_in_cycles'])
    X_seq, X_der, X_xgb, y, meta = [], [], [], [], []

    for unit in test_units:
        unit_b = test_b.loc[test_b['unit_number'] == unit].sort_values('time_in_cycles')
        assert len(unit_b) >= WINDOW_SIZE, (
            f'Test engine {unit} has {len(unit_b)} cycles; at least {WINDOW_SIZE} required'
        )

        last_cycle = int(unit_b['time_in_cycles'].iloc[-1])
        key = (unit, last_cycle)
        assert key in c_lookup.index

        endpoint_rul_raw = target_map[unit]
        endpoint_rul_capped = min(endpoint_rul_raw, float(RUL_CAP))

        X_seq.append(
            unit_b[SENSOR_COLS].iloc[-WINDOW_SIZE:].values.astype(np.float32)
        )
        X_der.append(
            c_lookup.loc[key, DERIVED_COLS].values.astype(np.float32)
        )
        X_xgb.append(
            c_lookup.loc[key, FEATURE_SET_C].values.astype(np.float32)
        )
        y.append(endpoint_rul_capped)
        meta.append({
            'unit_number': unit,
            'last_observed_cycle': last_cycle,
            'RUL_official_uncapped': endpoint_rul_raw,
            'RUL_capped': endpoint_rul_capped,
        })

    X_seq = np.asarray(X_seq, dtype=np.float32)
    X_der = np.asarray(X_der, dtype=np.float32)
    X_xgb = np.asarray(X_xgb, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    meta = pd.DataFrame(meta)

    assert X_seq.shape == (100, WINDOW_SIZE, len(SENSOR_COLS))
    assert X_der.shape == (100, len(DERIVED_COLS))
    assert X_xgb.shape == (100, len(FEATURE_SET_C))
    assert y.shape == (100,)
    assert len(meta) == 100 and meta['unit_number'].nunique() == 100
    assert meta['unit_number'].tolist() == list(range(1, 101))
    assert np.isfinite(X_seq).all() and np.isfinite(X_der).all()
    assert np.isfinite(X_xgb).all() and np.isfinite(y).all()

    return X_seq, X_der, X_xgb, y, meta


def calculate_endpoint_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.clip(np.asarray(y_pred, dtype=np.float64).reshape(-1), 0, RUL_CAP)
    error = y_pred - y_true

    return {
        'n_test_engines': int(len(y_true)),
        'rmse': float(root_mean_squared_error(y_true, y_pred)),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'median_absolute_error': float(median_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
        'mean_error': float(error.mean()),
        'error_std': float(error.std(ddof=1)),
        'min_error': float(error.min()),
        'max_error': float(error.max()),
    }


def run_official_test_evaluation():
    metrics_path = f'{FINAL_TEST_DIR}/final_test_endpoint_metrics_fd001.csv'
    predictions_path = f'{FINAL_TEST_DIR}/final_test_endpoint_predictions_fd001.csv'
    manifest_path = f'{FINAL_TEST_DIR}/official_test_evaluation_manifest.json'

    official_paths = [metrics_path, predictions_path, manifest_path]
    official_exists = [os.path.isfile(p) for p in official_paths]

    if any(official_exists) and not all(official_exists) and not OVERWRITE_OFFICIAL_TEST:
        existing = [p for p, e in zip(official_paths, official_exists) if e]
        missing  = [p for p, e in zip(official_paths, official_exists) if not e]
        raise RuntimeError(
            'Partial official-test artefacts detected. '
            'Refusing to rerun while OVERWRITE_OFFICIAL_TEST=False.\n'
            f'Existing: {existing}\nMissing: {missing}'
        )

    existing_complete = all(official_exists)

    if existing_complete and not OVERWRITE_OFFICIAL_TEST:
        print('Official-test outputs already exist and will not be overwritten.')
        metrics_df = pd.read_csv(metrics_path)
        predictions_df = pd.read_csv(predictions_path)
        assert len(metrics_df) == 4
        assert len(predictions_df) == 100
        return metrics_df, predictions_df

    if not AUTHORISE_OFFICIAL_TEST:
        print('OFFICIAL TEST NOT AUTHORISED.')
        print('Review Section 8, then set AUTHORISE_OFFICIAL_TEST=True in Section 2.')
        return None, None

    assert os.path.isfile(f'{FINAL_TEST_DIR}/pretest_freeze_manifest.json')
    print('=== AUTHORISED ONE-TIME OFFICIAL TEST EVALUATION ===')

    # Test features and labels are first accessed here.
    raw_test = pd.read_csv(
        f'{RAW_DIR}/test_FD001.txt',
        sep=r'\s+',
        header=None,
        names=ALL_COLS,
    )
    official_rul = pd.read_csv(
        f'{RAW_DIR}/RUL_FD001.txt',
        sep=r'\s+',
        header=None,
    ).iloc[:, 0].astype(float)

    frozen_scaler_b = joblib.load(f'{FULL_DATA_DIR}/scaler_feature_set_b_fd001.joblib')
    frozen_scaler_c = joblib.load(f'{FULL_DATA_DIR}/scaler_feature_set_c_fd001.joblib')

    X_seq_test, X_der_test, X_xgb_test, y_test, endpoint_meta = build_test_endpoints(
        raw_test,
        official_rul,
        frozen_scaler_b,
        frozen_scaler_c,
    )

    # Load the already-frozen model artefacts.
    xgb_model = joblib.load(MODEL_PATHS['XGBoost'])
    gru_model = tf.keras.models.load_model(MODEL_PATHS['GRU'])
    mlp_model = tf.keras.models.load_model(MODEL_PATHS['DerivedOnlyMLP'])
    fusion_model = tf.keras.models.load_model(MODEL_PATHS['MultiViewGRUFusion'])

    X_xgb_test_df = pd.DataFrame(X_xgb_test, columns=FEATURE_SET_C)

    predictions = {
        'XGBoost': np.clip(xgb_model.predict(X_xgb_test_df), 0, RUL_CAP),
        'GRU': np.clip(gru_model.predict(X_seq_test, verbose=0).ravel(), 0, RUL_CAP),
        'DerivedOnlyMLP': np.clip(mlp_model.predict(X_der_test, verbose=0).ravel(), 0, RUL_CAP),
        'MultiViewGRUFusion': np.clip(
            fusion_model.predict([X_seq_test, X_der_test], verbose=0).ravel(),
            0,
            RUL_CAP,
        ),
    }

    predictions_df = endpoint_meta.copy()
    metric_rows = []

    for model_name, pred in predictions.items():
        assert np.asarray(pred).shape == (100,)
        assert np.isfinite(pred).all()
        predictions_df[f'{model_name}_prediction'] = pred
        predictions_df[f'{model_name}_error'] = pred - y_test

        metrics = calculate_endpoint_metrics(y_test, pred)
        metric_rows.append({
            'model': model_name,
            **metrics,
            'target_definition': f'official endpoint RUL capped at {RUL_CAP}',
            'prediction_clipping': f'[0, {RUL_CAP}]',
        })

    metrics_df = pd.DataFrame(metric_rows).sort_values('rmse').reset_index(drop=True)

    assert len(predictions_df) == 100
    assert predictions_df['unit_number'].nunique() == 100
    assert not predictions_df['unit_number'].duplicated().any()
    assert set(metrics_df['model']) == set(predictions)

    predictions_df.to_csv(predictions_path, index=False)
    metrics_df.to_csv(metrics_path, index=False)

    evaluation_manifest = {
        'evaluated_at': datetime.now(timezone.utc).isoformat(),
        'authorised': True,
        'one_time_official_test_protocol': True,
        'test_engines': 100,
        'predictions_per_model': 100,
        'target': f'official endpoint RUL capped at {RUL_CAP}',
        'no_post_test_tuning_permitted': True,
        'final_epochs': FINAL_EPOCHS,
        'pretest_freeze_manifest_sha256': sha256_file(
            f'{FINAL_TEST_DIR}/pretest_freeze_manifest.json'
        ),
        'raw_test_sha256': sha256_file(f'{RAW_DIR}/test_FD001.txt'),
        'official_rul_sha256': sha256_file(f'{RAW_DIR}/RUL_FD001.txt'),
        'predictions_sha256': sha256_file(predictions_path),
        'metrics_sha256': sha256_file(metrics_path),
        'model_hashes': {
            model_name: sha256_file(model_path)
            for model_name, model_path in MODEL_PATHS.items()
        },
    }
    with open(manifest_path, 'w') as f:
        json.dump(evaluation_manifest, f, indent=2)

    del gru_model, mlp_model, fusion_model
    tf.keras.backend.clear_session(); gc.collect()

    print(f'Endpoint predictions saved: {predictions_path}')
    print(f'Endpoint metrics saved: {metrics_path}')
    print(f'Evaluation manifest saved: {manifest_path}')
    return metrics_df, predictions_df


print('Official-test construction and evaluation functions defined.')

## Section 10 — Execute or Load Official Test Evaluation

### Manual control

For the first execution:

1. Run Sections 0–8 with `AUTHORISE_OFFICIAL_TEST=False`.
2. Review `pretest_freeze_manifest.json` and confirm all four models are frozen.
3. Change `AUTHORISE_OFFICIAL_TEST=True` in Section 2.
4. Run this cell exactly once.
5. Do not alter models, epochs, features, scalers or hyperparameters after viewing the result.

On later notebook openings, existing official-test artefacts are loaded without replacement while `OVERWRITE_OFFICIAL_TEST=False`.

In [ ]:
final_test_metrics, final_test_predictions = run_official_test_evaluation()

if final_test_metrics is not None:
    print('\nOfficial endpoint metrics:')
    display_cols = ['model', 'rmse', 'mae', 'r2', 'mean_error', 'n_test_engines']
    print(final_test_metrics[display_cols].to_string(index=False))
else:
    print('\nOfficial test evaluation remains pending authorisation.')

## Section 11 — Interpretation Boundary

Complete the result interpretation only after the authorised execution. The interpretation must follow these rules:

- Report the official endpoint result for every frozen model, not only the best model.
- Treat the official test as a one-time held-out assessment, not a new tuning set.
- Do not change epoch counts, feature definitions, model architecture or hyperparameters after seeing the result.
- Compare the official-test ranking with NB10 repeated validation descriptively.
- Do not claim formal statistical superiority from one 100-engine test set.
- Preserve negative or unexpected results; they are part of the dissertation evidence.

## Section 12 — NB11 Completion Gate

In [ ]:
required_pretest = [
    f'{FINAL_TEST_DIR}/final_epoch_selection_fd001.csv',
    f'{FINAL_TEST_DIR}/pretest_freeze_manifest.json',
    f'{FINAL_TEST_DIR}/frozen_official_test_protocol.json',
    *MODEL_PATHS.values(),
    *HISTORY_PATHS.values(),
]
for path in required_pretest:
    assert os.path.isfile(path), f'Missing pre-test artefact: {path}'

required_official = [
    f'{FINAL_TEST_DIR}/final_test_endpoint_metrics_fd001.csv',
    f'{FINAL_TEST_DIR}/final_test_endpoint_predictions_fd001.csv',
    f'{FINAL_TEST_DIR}/official_test_evaluation_manifest.json',
]

official_complete = all(os.path.isfile(path) for path in required_official)

print('=== NB11 COMPLETION GATE ===')
print('[x] Final epochs derived from NB10 and frozen')
print('[x] All 100 training engines used for final model fitting')
print('[x] Scalers fitted only on full training data')
print('[x] Four frozen model artefacts saved')
print('[x] Pre-test freeze manifest saved before test-label access')

if official_complete:
    metrics_check = pd.read_csv(required_official[0])
    predictions_check = pd.read_csv(required_official[1])
    assert len(metrics_check) == 4
    assert len(predictions_check) == 100
    assert predictions_check['unit_number'].nunique() == 100
    print('[x] Exactly 100 official endpoint rows saved')
    print('[x] Metrics saved for all four models')
    print('[x] Official evaluation manifest saved')
    print('[x] No post-test tuning authorised')
    print('\nNB11 STATUS: COMPLETE — READY FOR REVIEW AND FREEZE')
else:
    print('[ ] Official test evaluation not yet authorised or completed')
    print('\nNB11 STATUS: PRE-TEST STAGE COMPLETE — OFFICIAL TEST PENDING')

## Section 13 — Artefact Listing

In [ ]:
for root in [FULL_DATA_DIR, FINAL_MODEL_DIR, FINAL_TEST_DIR]:
    print(f'\n{os.path.relpath(root, BASE)}/')
    for dirpath, _, filenames in os.walk(root):
        for filename in sorted(filenames):
            path = os.path.join(dirpath, filename)
            rel = os.path.relpath(path, BASE)
            print(f'  {rel}  ({os.path.getsize(path):,} bytes)')